# Bronze Layer - Data Ingestion

## Imports

In [120]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import pyarrow
import os

In [121]:
spark = (
    SparkSession.builder
    .appName("CarrierRiskPipeline")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

# Variables

In [122]:
datasets = {
    "communications": {
        "path": "../../samples/communication-samples.csv",
        "format": "csv",
        "expected_columns": [
            "external_id",
            "carrier_company_id",
            "broker_comany_id",
            "direction",
            "channel",
            "status",
            "created_at",
            "updated_at",
            "from_contact_type",
            "to_contact_type",
            "thread_id"
        ]
    },
    "brokers": {
        "path": "../../samples/broker-samples.txt",
        "format": "txt",
        "expected_columns": [
            "_c0",
            "_c1",
        ],
    },
    "carriers": {
        "path": "../../samples/carrier-samples.json",
        "format": "json",
        "expected_columns": [
            "id",
            "name",
        ],
    }
}

# path = "s3://bucket/communications/*.csv" # Reading from S3 bucket and if many files representing the same table

## Functions

In [123]:
def validate_schema(df, expected_columns):

    incoming_columns = set(df.columns)
    expected_columns = set(expected_columns)

    missing_columns = expected_columns - incoming_columns
    new_columns = incoming_columns - expected_columns

    if missing_columns:
        raise ValueError(
            f"Schema rejected. Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if new_columns:
        print(
            f"Schema evolution detected. "
            f"New columns: {sorted(new_columns)}"
        )

        return df

    print("Schema validation successful.")

    return df

In [124]:
def read_csv(spark, path, expected_columns):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(path)
    )

    return validate_schema(df, expected_columns)

In [125]:
def read_json(spark, path, expected_columns):

    df = (
        spark.read
        .option("multiLine", "true")
        .json(path)
    )

    validate_schema(df, expected_columns)

    return df

In [126]:
def read_txt(spark, path, expected_columns):
    df = (
        spark.read
        .option("sep", "\t")
        .option("inferSchema", "false")
        .csv(path)
    )

    return validate_schema(df, expected_columns)

In [127]:
def add_metadata(df):
    df_with_metadata = (
        df
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp())
    )

    return df_with_metadata

## Main

In [128]:
readers = {
    "csv": read_csv,
    "json": read_json,
    "txt": read_txt
}

In [129]:
for dataset_name, config in datasets.items():

    path = config["path"]
    file_format = config["format"]
    expected_columns = config.get("expected_columns")

    reader = readers.get(file_format)

    if reader is None:
        raise ValueError(
            f"Unsupported format: {file_format}"
        )

    df = reader(
        spark,
        path,
        expected_columns
    )

    df_with_metadata = add_metadata(df)

    output_path = f"../../data/bronze/{dataset_name}"
    # output_path = f"s3a://freighthero-data/bronze/{dataset_name}"

    os.makedirs(output_path, exist_ok=True)

    pandas_df = df_with_metadata.toPandas()

    pandas_df.to_parquet(
        f"{output_path}/data.parquet",
        index=False
    )

    print(
        f"Bronze dataset '{dataset_name}' saved successfully."
    )


    # Write Bronze dataset as Parquet
    # (
    #     df_with_metadata.write
    #     .mode("overwrite")
    #     .parquet(output_path)
    # )

    # print(
    #     f"Bronze dataset '{dataset_name}' successfully saved to: "
    #     f"{output_path}"
    # )

    # df_with_metadata.show(5, truncate=False)

Schema validation successful.


c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Bronze dataset 'communications' saved successfully.
Schema validation successful.


c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Bronze dataset 'brokers' saved successfully.
Schema validation successful.


c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Alex KM\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Bronze dataset 'carriers' saved successfully.
